In [2]:
import pandas as pd
import sqlite3

conn = sqlite3.connect("saas_data.db")

In [3]:
# --- Export 1: MAU trend ---
mau = pd.read_sql("""
    SELECT strftime('%Y-%m', event_date) AS month,
           COUNT(DISTINCT user_id) AS active_users
    FROM events GROUP BY month ORDER BY month
""", conn)
mau.to_csv("/Users/akshithreddy/Desktop/SaaS/Data/mau.csv", index=False)

In [4]:
# --- Export 2: Feature usage ---
features = pd.read_sql("""
    SELECT feature, COUNT(*) AS total_usage,
           COUNT(DISTINCT user_id) AS unique_users
    FROM events GROUP BY feature ORDER BY total_usage DESC
""", conn)
features.to_csv("/Users/akshithreddy/Desktop/SaaS/Data/feature_usage.csv", index=False)

In [5]:
# --- Export 3: Feature usage by plan ---
by_plan = pd.read_sql("""
    SELECT plan, feature, COUNT(*) AS usage_count
    FROM events GROUP BY plan, feature ORDER BY plan, usage_count DESC
""", conn)
by_plan.to_csv("/Users/akshithreddy/Desktop/SaaS/Data/usage_by_plan.csv", index=False)

In [6]:
# --- Export 4: User segments ---
event_counts = pd.read_sql("""
    SELECT user_id, COUNT(*) AS total_events,
           COUNT(DISTINCT feature) AS unique_features
    FROM events GROUP BY user_id
""", conn)
event_counts["engagement_score"] = (
    event_counts["total_events"] * 0.6 +
    event_counts["unique_features"] * 10 * 0.4
).round(1)

def segment(score):
    if score >= 80:   return "Power user"
    elif score >= 30: return "Casual user"
    else:             return "At-risk"

event_counts["segment"] = event_counts["engagement_score"].apply(segment)
event_counts.to_csv("/Users/akshithreddy/Desktop/SaaS/Data/user_segments.csv", index=False)

In [7]:
# --- Export 5: Feature adoption funnel ---
funnel_features = ["dashboard", "reports", "team_collab", "integrations", "api_access"]
funnel = pd.read_sql("""
    SELECT feature, COUNT(DISTINCT user_id) AS users
    FROM events WHERE feature IN ({})
    GROUP BY feature
""".format(",".join(f"'{f}'" for f in funnel_features)), conn)
funnel["order"] = funnel["feature"].map({f: i for i, f in enumerate(funnel_features)})
funnel = funnel.sort_values("order")
total = funnel["users"].iloc[0]
funnel["pct"] = (funnel["users"] / total * 100).round(1)
funnel.to_csv("/Users/akshithreddy/Desktop/SaaS/Data/funnel.csv", index=False)

In [16]:
print("All CSVs exported to /data folder!")
print("\nFiles ready for Tableau:")
for f in ["mau.csv","feature_usage.csv","usage_by_plan.csv","user_segments.csv","funnel.csv"]:
    print(f"  ✓ {f}")

All CSVs exported to /data folder!

Files ready for Tableau:
  ✓ mau.csv
  ✓ feature_usage.csv
  ✓ usage_by_plan.csv
  ✓ user_segments.csv
  ✓ funnel.csv
